In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from utils.helper import BatchControlHelper
from pyspark.sql import DataFrame
from pyspark.sql import functions as f
import json
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text(
    "table_metadata",
     "{'table_id': '11', 'table_name': 'customer_history', 'source_system': 'bronze', 'source_schema': 'data_source', 'source_table': 'customer_history', 'source_path': '', 'target_layer': 'silver', 'active_flag': 'True', 'load_order': '2', 'created_at': '2026-10-04 08:43:42.003751'}"
)
dbutils.widgets.text(
    "table_config",
    "{'load_type': 'MERGE', 'primary_key': 'customer_id,effective_from_date', 'watermark_column': 'source_modified_at'}"
)
dbutils.widgets.text("processing_date", "2024-12-31", "As-of date used when generating the initial load (YYYY-MM-DD)")
dbutils.widgets.dropdown("force_reload", "false", ["false", "true"], "Reprocess tables even if already SUCCESS")

processing_date = dbutils.widgets.get("processing_date")
force_reload = dbutils.widgets.get("force_reload") == "true"
env = dbutils.widgets.get("environment")
table_metadata = json.loads(dbutils.widgets.get("table_metadata").replace("'", '"'))
table_config = json.loads(dbutils.widgets.get("table_config").replace("'", '"'))

batch_id = f"ods-{processing_date.replace('-', '')}"

config = {
    "source_table": table_metadata["source_table"],
    "source_schema": table_metadata["source_schema"],
    "target_table": table_metadata["table_name"],
    "target_layer": table_metadata["target_layer"],
    "load_type": table_config["load_type"],
    "primary_key": table_config["primary_key"].split(','),
    "watermark_column": table_config["watermark_column"] if 'watermark_column' in table_config else None,
    'processing_date': processing_date,
    'force_reload': force_reload,
    'batch_id': batch_id,
    'env': env
}

config

In [0]:
source_specs = {
    "customer_history":{
        "source": "customer_history",
        "hash_key_name": "customer_hk",
        "business_keys": ["customer_id", "effective_from_date"],
        "columns": ["customer_id", "full_name", "address_line_1", "city", "state", "postal_code", "customer_segment", "kyc_status", "effective_from_date", "source_modified_at"]
    },
    "customer": {
        "source": "customer",
        "hash_key_name": "customer_hk",
        "business_keys": ["customer_id"],
        "columns": ["customer_id", "full_name", "address_line_1", "city", "state", "postal_code", "customer_segment", "kyc_status", "source_modified_at"],
    },
    "branch":{
        "source": "branch",
        "hash_key_name": "branch_hk",
        "business_keys": ["branch_id"],
        "columns": ["branch_id", "branch_name", "city", "state", "region", "source_modified_at"]
    },
    "account":{
        "source": "account",
        "hash_key_name": "account_hk",
        "business_keys": ["account_id"],
        "columns": ["account_id", "primary_customer_id", "branch_id", "account_type", "account_status", "open_date", "close_date", "currency_code", "source_modified_at"]
    },
    "product": {
        "source": "product",
        "hash_key_name": "product_hk",
        "business_keys": ["product_id"],
        "columns": ["product_id", "product_name", "product_category", "annual_fee"]
    },
    "account_customer_history":{
        "source": "account_customer_history",
        "hash_key_name": "ownership_hk",
        "business_keys": ["account_id", "customer_id", "effective_from_date"],
        "columns": ["account_id", "customer_id", "ownership_role", "ownership_percent", "effective_from_date", "source_modified_at"]
    },
    "transaction":{
        "source": "transaction",
        "hash_key_name": "transaction_hk",
        "business_keys": ["transaction_id"],
        "columns": ["transaction_id", "account_id", "transaction_timestamp", "transaction_date", "transaction_type", "transaction_amount", "currency_code", "merchant_name", "merchant_category", "transaction_status", "source_modified_at", "arrival_timestamp", "is_late_arrival"]
    },
    "transaction_line":{
        "source": "transaction_line",
        "hash_key_name": "transaction_line_hk",
        "business_keys": ["transaction_id", "line_number"],
        "columns": ["transaction_id", "line_number", "product_id", "quantity", "line_amount"]
    },
    "daily_account_balance":{
        "source": "daily_account_balance",
        "hash_key_name": "account_balance_hk",
        "business_keys": ["balance_date", "account_id"],
        "columns": ["balance_date", "account_id", "currency_code", "available_balance_amount", "ledger_balance_amount", "overdraft_limit_amount", "source_modified_at"]
    }
}

In [0]:
batch_controller = BatchControlHelper(
    spark=spark, 
    batch_control_table=f"{config['env']}_banking.metadata.batch_control",
    watermark_table=f"{config['env']}_banking.metadata.table_watermarks"
)

In [0]:
def get_work_data(table):
    previous_watermark = batch_controller.get_watermark(table, process_name='ods')
    spark.read.table(f'{env}_banking.silver.{table}').filter(
        f.col('source_modified_at') > previous_watermark).select('customer_id').dropDuplicates().createOrReplaceTempView('ods_work_table')
    return spark.sql(f"""
        SELECT t.* 
            FROM {env}_banking.silver.{table} t
        INNER JOIN (
            SELECT DISTINCT customer_id 
                FROM ods_work_table) wk
        ON t.customer_id = wk.customer_id
    """)

In [0]:
def ingest_silver_to_ods(config: dict):
    if force_reload:
        batch_controller.reset_watermark(target_table=config['target_table'], process_name=config['target_layer'])
    
    if not force_reload and batch_controller.has_succeeded(config['batch_id'], config['source_table']):
        print(f"SKIP    {config['source_table']}: already SUCCESS for batch {config['batch_id']}")
        return {"source_table": config['source_table'], "status": "SKIPPED", "record_count": None}
    
    run_id = batch_controller.start_batch(config['batch_id'], config['source_table'], config['target_table'], config['target_layer'], config['load_type'])
    print("RUN ID : ", run_id)
    try:
        source_spec = source_specs[config['source_table']]
        silver_table = f"{config['env']}_banking.silver.{config['source_table']}"
        version_window = None
        ods_table = f"{config['env']}_banking.ods.{config['target_table']}"
        if config['source_table'] == 'customer_history':
            silver_df = get_work_data(source_spec['source'])
            version_window = Window.partitionBy('customer_id').orderBy(f.col('effective_from_date').asc())
        elif config['source_table'] == 'account_customer_history':
            version_window = Window.partitionBy(['account_id', 'customer_id']).orderBy(f.col('effective_from_date').asc())
            silver_df = get_work_data(source_spec['source'])
        else:
            silver_df = spark.sql(f"SELECT * FROM {env}_banking.silver.{config['source_table']}")
            if not force_reload and config['load_type'] in ["APPEND", "MERGE"] and config['watermark_column']:
                previous_watermark = batch_controller.get_watermark(table_name=config['target_table'], process_name=config['target_layer'])
                silver_df = silver_df.filter(f.col(config['watermark_column']) > f.lit(previous_watermark))
        
        source_max_timestamp = (
            silver_df.select(f.max(f.col(config['watermark_column']))).first()[0] if config['watermark_column'] else None  # '2024-12-31'
        )
        silver_df = (
            silver_df.withColumn("processing_date", f.lit(config['processing_date']))
                .withColumn(source_spec['hash_key_name'], f.sha2(f.concat_ws('|', *source_spec['business_keys']), 256))
                .withColumn('row_hash_fingerprint', f.sha2(f.concat_ws('|', *source_spec['columns']), 256))
                .withColumn("_ingestion_run_id", f.lit(run_id))
                .withColumn("_ingestion_batch_id", f.lit(config['batch_id'])).drop('_next_effective_date')
        ) 
        if version_window:
            silver_df = (
                silver_df
                    .withColumn('record_version_number', f.row_number().over(version_window))
                    .withColumn('row_effective_date', f.to_date(f.col('effective_from_date')))
                    .withColumn('_next_effective_date', f.lead(f.to_date(f.col('effective_from_date'))).over(version_window))
                    .withColumn('row_expiration_date', f.col('_next_effective_date'))
                    .withColumn('is_current_version', f.col('_next_effective_date').isNull()).drop('_next_effective_date')
            ) 
        silver_df = silver_df.dropDuplicates(subset=['row_hash_fingerprint'])
        record_count = silver_df.count()  
        print("Records to process:", record_count) 
        if not spark.catalog.tableExists(ods_table) and record_count > 0:
            silver_df.write.format("delta").mode("overwrite").saveAsTable(ods_table)
        elif config['load_type'] == "FULL" and record_count > 0:
            print(f"FULL    {silver_table} -> {ods_table} Record count: {record_count}")
            silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(ods_table)
        elif config['load_type'] in ["APPEND", "MERGE"] and record_count > 0:
            print(f"MERGE/APPEND    {silver_table} -> {ods_table} Record count: {record_count}")
            if not config['primary_key']:
                raise Exception(f"Primary key is required for MERGE load type")
            delta_table = DeltaTable.forName(spark, ods_table)
            merge_condition = " AND ".join([f"bronze.{pk} = silver.{pk}" for pk in config['primary_key']])
            delta_table.alias("silver")\
                .merge(silver_df.alias("bronze"), merge_condition)\
                .whenMatchedUpdateAll()\
                .whenNotMatchedInsertAll()\
                .execute()

        if config['watermark_column']:
            batch_controller.update_watermark(config['target_layer'], config['target_table'], source_max_timestamp, run_id)
        
        batch_controller.complete_batch(run_id, record_count)
        print(
            f"SUCCESS {silver_table} -> {ods_table} "
            f"({record_count} new rows; watermark={config['watermark_column']})"
        )
        return {"source_table": config['source_table'], "status": "SUCCESS", "record_count": record_count}
    except Exception as error:
        batch_controller.fail_batch(run_id, str(error))
        print(f"FAILED  {config['source_table']}: {error}")
        raise Exception(error)

In [0]:
result = ingest_silver_to_ods(config)

In [0]:
# df = spark.read.table('dev_banking.silver.customer_history').filter("customer_id = 'C000008'")
# df = df.withColumn('state', f.lit('CA')).withColumn('customer_segment', f.lit('RETAIL')).withColumn('source_modified_at', f.lit('2024-07-28 08:00:00')).withColumn('effective_from_date', f.lit('2024-07-28 08:00:00'))
# df.show()#write.format("delta").mode("append").saveAsTable('dev_banking.silver.customer_history')

In [0]:
# source_df = spark.read.table('dev_banking.ods.customer_history')
# source_df.sort(['customer_id', 'effective_from_date', 'source_modified_at']).show()